# Pipeline: Sistema de Inteligência Territorial de Castanhal
**Repo**: https://github.com/LuanLindolfo/tcc

**Fluxo**: GitHub XLSX → Limpeza → Feature Engineering → ML → Push artefatos

> Configure `GITHUB_TOKEN` nos Colab Secrets antes de executar.

In [ ]:
# CÉLULA 1: Setup
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','pyarrow','xgboost','plotly','openpyxl'],check=True)
from google.colab import userdata
GITHUB_TOKEN  = userdata.get('GITHUB_TOKEN')
GITHUB_REPO   = 'LuanLindolfo/tcc'
GITHUB_BRANCH = 'main'
GITHUB_RAW    = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}'
XLSX_PATH     = 'censo_castanhal/censo_castanhal'
OUT_DIR       = '/content'
print(f'Setup OK — repo: {GITHUB_REPO}')

In [ ]:
# CÉLULA 2: Ingestão dos XLSX do GitHub
import pandas as pd, io, requests
from urllib.parse import quote

def ler_xlsx(nome):
    url = f"{GITHUB_RAW}/{XLSX_PATH}/{quote(nome)}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return pd.read_excel(io.BytesIO(r.content), dtype_backend='numpy_nullable')

ARQUIVOS = {
    'piramide_etaria':       'Censo 2022 - Pirâmide etária - Castanhal (PA).xlsx',
    'razao_sexo_envelh':     'Censo 2022 - Razão de sexo e índice de envelhecimento - Castanhal (PA).xlsx',
    'populacao_residente':   'Censo 2022 - População residente - Municípios.xlsx',
    'densidade_demografica': 'Censo 2022 - Densidade demográfica - Municípios.xlsx',
    'taxa_crescimento':      'Censo 2022 - Taxa de crescimento anual da população - Municípios.xlsx',
    'caract_domicilios':     'Censo 2022 - Características dos domicílios - Castanhal (PA).xlsx',
    'condicoes_ocupacao':    'Censo 2022 - Condições de ocupação - Castanhal (PA).xlsx',
    'tipos_domicilio':       'Censo 2022 - Tipos de domicílio - Castanhal (PA).xlsx',
    'material_paredes':      'Censo 2022 - Tipo de material das paredes externas - Castanhal (PA).xlsx',
    'media_moradores':       'Censo 2022 - Média de moradores por domicílio - Castanhal (PA).xlsx',
    'alfabetizacao':         'Censo 2022 - Alfabetização - Castanhal (PA).xlsx',
    'nivel_instrucao':       'Censo 2022 - Nível de instrução - Castanhal (PA).xlsx',
    'freq_escolar':          'Censo 2022 - Taxa bruta de frequência escolar, por grupo de idade - Castanhal (PA).xlsx',
    'escolaridade_homens':   'analise_escolaridade_homens_percetual.xlsx',
    'escolaridade_mulheres': 'analise_escolaridade_mulheres_percetual.xlsx',
    'rendimento_percapita':  'Censo 2022 - Rendimento domiciliar mensal per capita - Castanhal (PA).xlsx',
    'dist_renda':            'distribuição de renda.xlsx',
    'profissoes':            'profissões.xlsx',
    'taxa_atividade_pct':    'taxa_atividade_percentual.xlsx',
}

print('Baixando arquivos...')
dados_raw = {}
erros = []
for chave, arquivo in ARQUIVOS.items():
    try:
        dados_raw[chave] = ler_xlsx(arquivo)
        print(f'  OK {chave}: {dados_raw[chave].shape}')
    except Exception as e:
        erros.append(chave)
        print(f'  ERRO {chave}: {e}')
print(f'Ingestao: {len(dados_raw)}/{len(ARQUIVOS)} OK')

In [ ]:
# CÉLULA 3: Limpeza
import unicodedata, re, numpy as np

def snake(s):
    s = unicodedata.normalize('NFKD',str(s)).encode('ascii','ignore').decode()
    s = re.sub(r'[^\w\s]','',s).strip().lower()
    return re.sub(r'\s+','_',s)

def limpar(df):
    df = df.copy()
    df.columns = [snake(c) for c in df.columns]
    num = df.select_dtypes(include='number').columns
    df[num] = df[num].fillna(df[num].median())
    return df

dados = {k: limpar(v) for k,v in dados_raw.items()}
for k,df in dados.items():
    print(f'{k}: {list(df.columns[:6])}')
print('Limpeza OK')

In [ ]:
# CÉLULA 4: Consolidação
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

def concat(*keys):
    frames = [dados[k] for k in keys if k in dados and not dados[k].empty]
    return pd.concat(frames, axis=0, ignore_index=True) if frames else pd.DataFrame()

df_demografico    = concat('piramide_etaria','razao_sexo_envelh','populacao_residente','densidade_demografica')
df_domicilios     = concat('caract_domicilios','condicoes_ocupacao','tipos_domicilio','material_paredes','media_moradores')
df_educacao       = concat('alfabetizacao','nivel_instrucao','freq_escolar','escolaridade_homens','escolaridade_mulheres')
df_trabalho_renda = concat('rendimento_percapita','dist_renda','profissoes','taxa_atividade_pct')

# IAH
def calc_iah(df):
    cols = [c for c in df.columns if any(x in c for x in ['agua','esgoto','lixo','energia','saneamento'])]
    return (df[cols].mean(axis=1)/100).clip(0,1) if cols else pd.Series([0.5]*len(df))

df_features = df_domicilios.copy()
df_features['iah'] = calc_iah(df_features)

# IVS via KMeans
num_cols = df_features.select_dtypes(include='number').columns.tolist()
if len(num_cols) >= 2 and len(df_features) > 6:
    X = MinMaxScaler().fit_transform(df_features[num_cols].fillna(0))
    km = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    scores = X.mean(axis=1)
    means = {i: scores[labels==i].mean() for i in range(3)}
    ordem = sorted(means, key=lambda k: means[k])
    nomes = {ordem[0]:'baixa', ordem[1]:'media', ordem[2]:'alta'}
    df_features['ivs_label'] = [nomes[l] for l in labels]
    df_features['cluster_ocupacao'] = labels
else:
    df_features['ivs_label'] = 'media'
    df_features['cluster_ocupacao'] = 0

print(f'demografico: {df_demografico.shape}')
print(f'domicilios: {df_domicilios.shape}')
print(f'educacao: {df_educacao.shape}')
print(f'trabalho_renda: {df_trabalho_renda.shape}')
print(f'features: {df_features.shape}')

In [ ]:
# CÉLULA 5: EDA
import plotly.express as px
if 'iah' in df_features.columns:
    fig = px.histogram(df_features, x='iah', color='ivs_label', title='IAH por IVS')
    fig.show()
print('EDA OK')

In [ ]:
# CÉLULA 6: Classificação IVS
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

modelo_clf, metricas_clf = None, {}
if 'ivs_label' in df_features.columns and len(df_features) > 10:
    excluir = ['ivs_label','cluster_ocupacao','iah']
    feat = [c for c in df_features.select_dtypes(include='number').columns if c not in excluir]
    X = df_features[feat].fillna(df_features[feat].median())
    y = df_features['ivs_label']
    Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
    clf = RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1)
    clf.fit(Xtr,ytr)
    yp = clf.predict(Xte)
    modelo_clf = clf
    metricas_clf = {'modelo':'RandomForestClassifier','data_treino':pd.Timestamp.now().strftime('%Y-%m-%d'),
                    'acuracia':round(accuracy_score(yte,yp),4),'f1_macro':round(f1_score(yte,yp,average='macro'),4),
                    'classes':sorted(y.unique().tolist()),'matriz_confusao':confusion_matrix(yte,yp).tolist(),
                    'feature_importances':dict(sorted(zip(feat,clf.feature_importances_),key=lambda x:x[1],reverse=True)[:15])}
    print(f'RF Acuracia={metricas_clf["acuracia"]:.2%} F1={metricas_clf["f1_macro"]:.2%}')
else:
    print('Dados insuficientes para classificacao')

In [ ]:
# CÉLULA 7: Regressão IAH
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

modelo_reg, metricas_reg = None, {}
if 'iah' in df_features.columns and len(df_features) > 10:
    excluir = ['iah','ivs_label','cluster_ocupacao']
    feat = [c for c in df_features.select_dtypes(include='number').columns if c not in excluir]
    X = df_features[feat].fillna(df_features[feat].median())
    y = df_features['iah']
    Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42)
    reg = XGBRegressor(n_estimators=200,random_state=42,learning_rate=0.05,max_depth=4,n_jobs=-1)
    reg.fit(Xtr,ytr)
    yp = reg.predict(Xte)
    modelo_reg = reg
    metricas_reg = {'modelo':'XGBRegressor','data_treino':pd.Timestamp.now().strftime('%Y-%m-%d'),
                    'r2':round(r2_score(yte,yp),4),'rmse':round(float(np.sqrt(mean_squared_error(yte,yp))),4),
                    'mae':round(float(mean_absolute_error(yte,yp)),4),'y_test':yte.tolist(),'y_pred':yp.tolist()}
    print(f'XGB R2={metricas_reg["r2"]:.4f} RMSE={metricas_reg["rmse"]:.4f}')
else:
    print('Dados insuficientes para regressao')

In [ ]:
# CÉLULA 8: Clustering
from sklearn.cluster import KMeans as KM
modelo_km = None
feat_km = [c for c in df_features.select_dtypes(include='number').columns
           if any(x in c for x in ['primario','secundario','terciario','renda','rendimento'])]
if feat_km and len(df_features) > 6:
    km2 = KM(n_clusters=min(3,len(df_features)//2),random_state=42,n_init=10)
    df_features['cluster_ocupacao'] = km2.fit_predict(df_features[feat_km].fillna(0))
    modelo_km = km2
    print(f'KMeans OK features={feat_km}')
else:
    print('Clustering fallback (sem colunas de ocupacao)')

In [ ]:
# CÉLULA 9: Exportação
import joblib, json, os

for p in ['data/processed','data/results','models']:
    os.makedirs(f'{OUT_DIR}/{p}', exist_ok=True)

df_demografico.to_parquet(f'{OUT_DIR}/data/processed/demografico.parquet',index=False)
df_domicilios.to_parquet(f'{OUT_DIR}/data/processed/domicilios.parquet',index=False)
df_educacao.to_parquet(f'{OUT_DIR}/data/processed/educacao.parquet',index=False)
df_trabalho_renda.to_parquet(f'{OUT_DIR}/data/processed/trabalho_renda.parquet',index=False)
df_features.to_parquet(f'{OUT_DIR}/data/processed/features_compostas.parquet',index=False)
print('Parquets OK')

if modelo_clf: joblib.dump(modelo_clf,f'{OUT_DIR}/models/random_forest_ivs.joblib')
if modelo_reg: joblib.dump(modelo_reg,f'{OUT_DIR}/models/xgboost_iah.joblib')
if modelo_km:  joblib.dump(modelo_km, f'{OUT_DIR}/models/kmeans_ocupacao.joblib')
print('Modelos OK')

with open(f'{OUT_DIR}/data/results/ml_classificacao_metricas.json','w',encoding='utf-8') as f:
    json.dump(metricas_clf,f,ensure_ascii=False,indent=2)
with open(f'{OUT_DIR}/data/results/ml_regressao_metricas.json','w',encoding='utf-8') as f:
    json.dump(metricas_reg,f,ensure_ascii=False,indent=2)

politicas = [
    {'politica':'Habitação Popular','area':'domicilios','modelo_relacionado':'xgboost_iah','indicador_chave':'iah','limiar':0.5,'descricao':'Setores com IAH < 0,5 têm déficit habitacional crítico.','recomendacao':'Mapear domicílios sem saneamento para programas emergenciais.'},
    {'politica':'Combate ao Analfabetismo','area':'educacao','modelo_relacionado':'random_forest_ivs','indicador_chave':'taxa_analfabetismo','limiar':None,'descricao':'Setores acima do Q3 de analfabetismo demandam EJA.','recomendacao':'Expandir EJA nos setores de alta vulnerabilidade.'},
    {'politica':'Geração de Emprego & Renda','area':'trabalho_renda','modelo_relacionado':'kmeans_ocupacao','indicador_chave':'cluster_ocupacao','limiar':None,'descricao':'Clusters primários indicam necessidade de qualificação.','recomendacao':'Cursos técnicos e microcrédito nos clusters de baixa renda.'},
    {'politica':'Saúde & Envelhecimento Ativo','area':'demografico','modelo_relacionado':'random_forest_ivs','indicador_chave':'zona_envelhecimento','limiar':None,'descricao':'Zonas de envelhecimento requerem infraestrutura geriátrica.','recomendacao':'Ampliar UBSs com geriatria e centros de convivência.'},
    {'politica':'Infraestrutura Urbana Integrada','area':'domicilios','modelo_relacionado':'xgboost_iah','indicador_chave':'iah','limiar':0.7,'descricao':'IAH entre 0,5 e 0,7 indica necessidade de melhorias incrementais.','recomendacao':'Priorizar asfalto e coleta conforme o Plano Diretor.'}
]
with open(f'{OUT_DIR}/data/results/politicas_recomendacoes.json','w',encoding='utf-8') as f:
    json.dump(politicas,f,ensure_ascii=False,indent=2)
print('Exportacao completa')

In [ ]:
# CÉLULA 10: Push para GitHub
import subprocess, shutil

REPO_LOCAL = '/tmp/tcc_push'
repo_url_auth = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git'

if os.path.exists(REPO_LOCAL): shutil.rmtree(REPO_LOCAL)

subprocess.run(['git','config','--global','user.email','pipeline@tcc.local'],check=True)
subprocess.run(['git','config','--global','user.name','TCC Pipeline'],check=True)
subprocess.run(['git','clone','--depth','1',f'https://github.com/{GITHUB_REPO}.git',REPO_LOCAL],check=True)

for pasta in ['data/processed','data/results','models']:
    dst = f'{REPO_LOCAL}/{pasta}'
    os.makedirs(dst,exist_ok=True)
    src = f'{OUT_DIR}/{pasta}'
    for fname in os.listdir(src):
        shutil.copy2(f'{src}/{fname}',f'{dst}/{fname}')

subprocess.run(['git','-C',REPO_LOCAL,'add','data/processed/','data/results/','models/'],check=True)
result = subprocess.run(['git','-C',REPO_LOCAL,'commit','-m','chore: update processed data and models [pipeline]'],capture_output=True,text=True)
print(result.stdout or 'nada novo')

subprocess.run(['git','-C',REPO_LOCAL,'remote','set-url','origin',repo_url_auth],check=True)
push = subprocess.run(['git','-C',REPO_LOCAL,'push','origin',GITHUB_BRANCH],capture_output=True,text=True)
if push.returncode == 0:
    print(f'Push OK! https://github.com/{GITHUB_REPO}/tree/{GITHUB_BRANCH}/data/')
else:
    print(f'Erro push: {push.stderr}')